In [2]:
# ==========================================================
# DAY 21 - POWER BI DATA PREPARATION
# ==========================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

INPUT_FILE = PROJECT_ROOT / "03_processed_data/supplier_final_evaluation.csv"
OUTPUT_FOLDER = PROJECT_ROOT / "05_powerbi_data"

OUTPUT_FOLDER.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_FILE)

print("Dataset Loaded:", df.shape)

# ----------------------------------------------------------
# 1. CLEAN & SELECT FINAL COLUMNS
# ----------------------------------------------------------

final_df = df[[
    "vendor_name",
    "total_quantity",
    "avg_cost",
    "avg_defect",
    "efficiency",
    "reliability_score",
    "risk_score",
    "spoilage_rate",
    "performance_score",
    "performance_rank",
    "final_category",
    "business_decision"
]]

# Save main dataset
final_df.to_csv(OUTPUT_FOLDER / "dashboard_main.csv", index=False)

# ----------------------------------------------------------
# 2. KPI DATASET (FOR CARDS)
# ----------------------------------------------------------

kpi = {
    "Total Vendors": len(df),
    "Top Performers": len(df[df["final_category"] == "Top Performer"]),
    "Critical Suppliers": len(df[df["final_category"] == "Critical Risk"]),
    "Average Cost": df["avg_cost"].mean(),
    "Average Defect": df["avg_defect"].mean(),
    "Average Performance Score": df["performance_score"].mean()
}

kpi_df = pd.DataFrame(list(kpi.items()), columns=["Metric", "Value"])
kpi_df.to_csv(OUTPUT_FOLDER / "kpi_cards.csv", index=False)

# ----------------------------------------------------------
# 3. CATEGORY DISTRIBUTION
# ----------------------------------------------------------

category_dist = df["final_category"].value_counts().reset_index()
category_dist.columns = ["Category", "Count"]

category_dist.to_csv(
    OUTPUT_FOLDER / "category_distribution.csv",
    index=False
)

# ----------------------------------------------------------
# 4. TOP 10 & BOTTOM 10 (FOR BAR CHARTS)
# ----------------------------------------------------------

top10 = df.sort_values("performance_score", ascending=False).head(10)
bottom10 = df.sort_values("performance_score").head(10)

top10.to_csv(OUTPUT_FOLDER / "top10_vendors.csv", index=False)
bottom10.to_csv(OUTPUT_FOLDER / "bottom10_vendors.csv", index=False)

# ----------------------------------------------------------
# 5. RISK DISTRIBUTION
# ----------------------------------------------------------
# ----------------------------------------------------------
# CREATE RISK CATEGORY (FIX)
# ----------------------------------------------------------

def risk_category(score):
    if score >= 0.7:
        return "High Risk"
    elif score >= 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_category"] = df["risk_score"].apply(risk_category)

risk_dist = df["risk_category"].value_counts().reset_index()
risk_dist.columns = ["Risk Category", "Count"]

risk_dist.to_csv(
    OUTPUT_FOLDER / "risk_distribution.csv",
    index=False
)

# ----------------------------------------------------------
# 6. SCATTER PLOT DATA (COST vs DEFECT)
# ----------------------------------------------------------

scatter_df = df[[
    "vendor_name",
    "avg_cost",
    "avg_defect",
    "final_category"
]]

scatter_df.to_csv(
    OUTPUT_FOLDER / "cost_defect_scatter.csv",
    index=False
)

# ----------------------------------------------------------
# 7. DECISION SUMMARY
# ----------------------------------------------------------

decision_summary = df["business_decision"].value_counts().reset_index()
decision_summary.columns = ["Decision", "Count"]

decision_summary.to_csv(
    OUTPUT_FOLDER / "decision_summary.csv",
    index=False
)

# ----------------------------------------------------------
# 8. SAVE FINAL CHECK FILE
# ----------------------------------------------------------

df.to_csv(OUTPUT_FOLDER / "final_full_dataset.csv", index=False)

print("\nAll Power BI datasets created successfully")

Dataset Loaded: (20, 25)

All Power BI datasets created successfully
